In [ ]:
%%writefile lua/rate_limiter.lua
local _M = {}
local redis = require "resty.redis"

function _M.check_rate(custom_capacity, custom_refill_rate)
    local client_ip = ngx.var.remote_addr
    if not client_ip or client_ip == "" then
        client_ip = "global"
    end
    
    local BUCKET_CAPACITY = tonumber(custom_capacity) or 5
    local REFILL_RATE = tonumber(custom_refill_rate) or 1
    
    -- 1. Inicializar la conexión a Redis
    local red = redis:new()
    red:set_timeouts(1000, 1000, 1000) -- 1 segundo max de espera (¡No queremos ralentizar el Gateway!)
    
    -- Nos conectamos al contenedor de Redis (usaremos el nombre del servicio en la red de Docker)
    local ok, err = red:connect("mi-redis-brain", 6379)
    if not ok then
        ngx.log(ngx.ERR, "[Gateway Fail-Safe] No se pudo conectar a Redis: ", err)
        return -- Si Redis cae, dejamos pasar el tráfico por seguridad (Fail-safe)
    end
    
    local tokens_key = client_ip .. ":tokens"
    local time_key = client_ip .. ":last_time"
    local now = ngx.now()
    
    -- 2. Traer los datos desde la memoria centralizada de Redis
    -- Usamos pipeline para hacer ambas consultas en un solo viaje de red
    red:init_pipeline()
    red:get(tokens_key)
    red:get(time_key)
    local results, err = red:commit_pipeline()
    
    if not results then
        ngx.log(ngx.ERR, "[Gateway] Error en pipeline de Redis: ", err)
        return
    end
    
    local tokens = results[1]
    local last_time = results[2]
    
    -- Si el dato regresa como "null" de Redis, Lua lo lee como ngx.null
    if tokens == ngx.null or not tokens then
        tokens = BUCKET_CAPACITY
        last_time = now
    else
        tokens = tonumber(tokens)
        last_time = tonumber(last_time)
        
        local time_passed = now - last_time
        local tokens_to_add = time_passed * REFILL_RATE
        tokens = math.min(BUCKET_CAPACITY, tokens + tokens_to_add)
    end
    
    -- 3. Evaluar el freno de mano
    if tokens >= 1 then
        tokens = tokens - 1
        
        -- Guardamos los nuevos valores en Redis con un tiempo de vida (TTL) de 1 hora
        -- Así, si una IP deja de atacar, Redis limpia la memoria solo (Adiós problemas de llenado)
        red:init_pipeline()
        red:setex(tokens_key, 3600, tokens)
        red:setex(time_key, 3600, now)
        red:commit_pipeline()
        
        -- Devolver la conexión al pool para reutilizarla (Optimización Alta)
        red:set_keepalive(10000, 100)
        return
    else
        -- Registrar el bloqueo en los logs centrales
        ngx.log(ngx.WARN, "[Gateway Freno de Mano CENTRAL] IP Bloqueada: ", client_ip)
        
        ngx.status = ngx.HTTP_TOO_MANY_REQUESTS
        ngx.header.content_type = "application/json; charset=utf-8"
        ngx.say([[{"error": "Too Many Requests", "message": "Freno de mano: Has excedido el límite centralizado del Gateway."}]])
        
        red:set_keepalive(10000, 100)
        return ngx.exit(ngx.HTTP_OK)
    end
end

return _M

In [2]:
# 1. Crear una red dedicada para nuestro producto
!docker network create gateway-network 2>/dev/null || true

# 2. Apagar el contenedor viejo
!docker rm -f mi-openresty-gateway mi-redis-brain 2>/dev/null || true

# 3. Levantar el "cerebro" (Redis) en nuestra red
!docker run -d \
  --name mi-redis-brain \
  --network gateway-network \
  redis:alpine

# 4. Levantar nuestro OpenResty conectado a la misma red
!docker run -d \
  --name mi-openresty-gateway \
  --network gateway-network \
  -p 8080:8080 \
  -v "$(pwd)/conf/nginx.conf:/usr/local/openresty/nginx/conf/nginx.conf:ro" \
  -v "$(pwd)/lua:/usr/local/openresty/nginx/lua:ro" \
  openresty/openresty:alpine

69604a1130c2d0fa731c5a0879a31ebd8dfc8330533c5d4fcb8afbdc21f183e5
mi-openresty-gateway
Unable to find image 'redis:alpine' locally
alpine: Pulling from library/redis

1ffc071e: Already exists 
d4ffd5c9: Pulling fs layer 
38c295a8: Pulling fs layer 
100e7e68: Pulling fs layer 
29b74aab: Pulling fs layer 
b700ef54: Pulling fs layer 
Digest: sha256:9d317178eceac8454a2284a9e6df2466b93c745529947f0cd42a0fa9609d7005[5A
Status: Downloaded newer image for redis:alpine
8d022894a8ddca1d78b7968079149d7a61a743610371d195efeb3402c424761f
3d3b9bfce11d672a45b45fcdffd308373143a6b5f5b160b09c32fae8e315d65c
docker: Error response from daemon: failed to create shim task: OCI runtime create failed: runc create failed: unable to start container process: error during container init: error mounting "/host_mnt/Users/admin/Desktop/Shafer_Python_Classes/Cloud_engineering_practice/optimizacion_de_servidor/conf/nginx.conf" to rootfs at "/usr/local/openresty/nginx/conf/nginx.conf": mount /host_mnt/Users/admin/Desktop/

In [4]:
# 1. Aseguramos que el archivo nginx.conf exista de verdad como archivo
%%writefile conf/nginx.conf
worker_processes 1;
events { worker_connections 1024; }
http {
    lua_package_path "/usr/local/openresty/nginx/lua/?.lua;;";
    server {
        listen 8080;
        server_name localhost;
        lua_code_cache off;
        location /api {
            set $limit_bucket_capacity 5;
            set $limit_refill_rate 2;
            content_by_lua_block {
                package.loaded["rate_limiter"] = nil
                local limiter = require "rate_limiter"
                limiter.check_rate(ngx.var.limit_bucket_capacity, ngx.var.limit_refill_rate)
                ngx.status = ngx.HTTP_OK
                ngx.header.content_type = "application/json; charset=utf-8"
                ngx.say([[{"status": "success", "message": "Petición procesada: Tráfico limpio a través del Gateway centralizado"}]] )
            }
        }
    }
}

# 2. Forzamos el borrado de cualquier contenedor trancado
!docker rm -f mi-openresty-gateway 2>/dev/null || true

# 3. Lanzamos Docker usando la variable PWD explícita de tu Mac para evitar confusiones de ruta
!docker run -d \
  --name mi-openresty-gateway \
  --network gateway-network \
  -p 8080:8080 \
  -v "$(pwd)/conf/nginx.conf:/usr/local/openresty/nginx/conf/nginx.conf:ro" \
  -v "$(pwd)/lua:/usr/local/openresty/nginx/lua:ro" \
  openresty/openresty:alpine

SyntaxError: invalid syntax (2301131136.py, line 3)

In [6]:
# 1. Borramos la carpeta falsa 'nginx.conf' creada por Docker en tu Mac
!rm -rf conf/nginx.conf

# 2. Creamos un archivo vacío real para asegurar el tiro
!touch conf/nginx.conf

In [7]:
%%writefile conf/nginx.conf
worker_processes 1;
events { worker_connections 1024; }
http {
    lua_package_path "/usr/local/openresty/nginx/lua/?.lua;;";
    server {
        listen 8080;
        server_name localhost;
        lua_code_cache off;
        location /api {
            set $limit_bucket_capacity 5;
            set $limit_refill_rate 2;
            content_by_lua_block {
                package.loaded["rate_limiter"] = nil
                local limiter = require "rate_limiter"
                limiter.check_rate(ngx.var.limit_bucket_capacity, ngx.var.limit_refill_rate)
                ngx.status = ngx.HTTP_OK
                ngx.header.content_type = "application/json; charset=utf-8"
                ngx.say([[{"status": "success", "message": "Petición procesada: Tráfico limpio a través del Gateway centralizado"}]] )
            }
        }
    }
}

Overwriting conf/nginx.conf


In [8]:
!docker rm -f mi-openresty-gateway 2>/dev/null || true

!docker run -d \
  --name mi-openresty-gateway \
  --network gateway-network \
  -p 8080:8080 \
  -v "$(pwd)/conf/nginx.conf:/usr/local/openresty/nginx/conf/nginx.conf:ro" \
  -v "$(pwd)/lua:/usr/local/openresty/nginx/lua:ro" \
  openresty/openresty:alpine

mi-openresty-gateway
f2bc40e6c368c294c38934d0acaf326d05667f9507566df5731f2005ad9dcd6d


In [9]:
!for i in {1..6}; do curl -s -o /dev/null -w "%{http_code}\n" http://localhost:8080/api; done

500
500
500
500
500
500


In [ ]:
!docker logs mi-openresty-gateway | tail -n 20

In [11]:
%%writefile conf/nginx.conf
worker_processes 1;
events { worker_connections 1024; }
http {
    # Agregamos './?.lua' para que busque en el directorio actual de ejecución
    lua_package_path "./?.lua;/usr/local/openresty/nginx/lua/?.lua;;" ;
    
    server {
        listen 8080;
        server_name localhost;
        lua_code_cache off;
        
        location /api {
            set $limit_bucket_capacity 5;
            set $limit_refill_rate 2;
            content_by_lua_block {
                package.loaded["rate_limiter"] = nil
                local limiter = require "rate_limiter"
                limiter.check_rate(ngx.var.limit_bucket_capacity, ngx.var.limit_refill_rate)
                ngx.status = ngx.HTTP_OK
                ngx.header.content_type = "application/json; charset=utf-8"
                ngx.say([[{"status": "success", "message": "Petición procesada: Tráfico limpio a través del Gateway centralizado"}]] )
            }
        }
    }
}

Writing conf/nginx.conf


FileNotFoundError: [Errno 2] No such file or directory: 'conf/nginx.conf'

In [12]:
import os

# Aseguramos que la carpeta exista en tu directorio actual
os.makedirs("config", exist_ok=True)

print("¡Carpeta 'config' verificada y lista!")

¡Carpeta 'config' verificada y lista!


In [13]:
%%writefile config/nginx.conf
worker_processes 1;
events { worker_connections 1024; }
http {
    # Buscamos en el directorio raíz del contenedor y en la carpeta lua
    lua_package_path "./?.lua;/usr/local/openresty/nginx/lua/?.lua;;" ;
    
    server {
        listen 8080;
        server_name localhost;
        lua_code_cache off;
        
        location /api {
            set $limit_bucket_capacity 5;
            set $limit_refill_rate 2;
            content_by_lua_block {
                package.loaded["rate_limiter"] = nil
                local limiter = require "rate_limiter"
                limiter.check_rate(ngx.var.limit_bucket_capacity, ngx.var.limit_refill_rate)
                ngx.status = ngx.HTTP_OK
                ngx.header.content_type = "application/json; charset=utf-8"
                ngx.say([[{"status": "success", "message": "Petición procesada: Tráfico limpio a través del Gateway centralizado"}]] )
            }
        }
    }
}

Writing config/nginx.conf


In [14]:
# Limpiamos el contenedor anterior si quedó colgado
!docker stop mi-openresty-gateway
!docker rm mi-openresty-gateway

# Levantamos usando tus carpetas reales del workspace
!docker run -d --name mi-openresty-gateway \
  --network mi-red-local \
  -p 8080:8080 \
  -v "$(pwd)/config:/usr/local/openresty/nginx/conf" \
  -v "$(pwd)/lua:/usr/local/openresty/nginx" \
  openresty/openresty:alpine

mi-openresty-gateway
mi-openresty-gateway
12d6f7f6fe56cf5876f3b3b36cb9583fc48df97e045a4b601bfaa5ed2c10d2da
docker: Error response from daemon: network mi-red-local not found.


In [15]:
# 1. Eliminamos el contenedor fallido
!docker rm -f mi-openresty-gateway 2>/dev/null || true

# 2. Levantamos OpenResty en la red correcta: gateway-network
!docker run -d --name mi-openresty-gateway \
  --network gateway-network \
  -p 8080:8080 \
  -v "$(pwd)/config:/usr/local/openresty/nginx/conf" \
  -v "$(pwd)/lua:/usr/local/openresty/nginx" \
  openresty/openresty:alpine

mi-openresty-gateway
16c991aa43c60a767787dfab3d1b71e65b5e13a8a55882e40396c5e4e82b9a7a
docker: Error response from daemon: failed to create shim task: OCI runtime create failed: runc create failed: unable to start container process: exec: "/usr/local/openresty/bin/openresty": stat /usr/local/openresty/bin/openresty: no such file or directory: unknown.


In [16]:
# 1. Limpiamos el contenedor roto
!docker rm -f mi-openresty-gateway 2>/dev/null || true

# 2. Levantamos con montajes aislados y específicos a nivel de archivo/subcarpeta
!docker run -d --name mi-openresty-gateway \
  --network gateway-network \
  -p 8080:8080 \
  -v "$(pwd)/config/nginx.conf:/usr/local/openresty/nginx/conf/nginx.conf:ro" \
  -v "$(pwd)/lua:/usr/local/openresty/nginx/html" \
  openresty/openresty:alpine

mi-openresty-gateway
6c8c79f4f82668a44069addf34d32217ffc75f44a082f4de119550226eb2022a


In [17]:
%%writefile config/nginx.conf
worker_processes 1;
events { worker_connections 1024; }
http {
    # Apuntamos el package_path a la carpeta html donde mapeamos tu script lua
    lua_package_path "/usr/local/openresty/nginx/html/?.lua;;";
    
    server {
        listen 8080;
        server_name localhost;
        lua_code_cache off;
        
        location /api {
            set $limit_bucket_capacity 5;
            set $limit_refill_rate 2;
            content_by_lua_block {
                package.loaded["rate_limiter"] = nil
                local limiter = require "rate_limiter"
                limiter.check_rate(ngx.var.limit_bucket_capacity, ngx.var.limit_refill_rate)
                ngx.status = ngx.HTTP_OK
                ngx.header.content_type = "application/json; charset=utf-8"
                ngx.say([[{"status": "success", "message": "Petición procesada: Tráfico limpio a través del Gateway centralizado"}]] )
            }
        }
    }
}

Overwriting config/nginx.conf


In [18]:
!docker restart mi-openresty-gateway

mi-openresty-gateway


In [19]:
!for i in {1..6}; do curl -s -o /dev/null -w "%{http_code}\n" http://localhost:8080/api; done

500
500
500
500
500
500


In [ ]:
!docker logs mi-openresty-gateway | tail -n 20

In [21]:
%%writefile config/nginx.conf
worker_processes 1;
events { worker_connections 1024; }

http {
    # Agregamos el DNS interno de Docker para que Nginx pueda resolver el nombre "mi-redis-brain"
    resolver 127.0.0.11 valid=10s;

    server {
        listen 8080;
        server_name localhost;
        lua_code_cache off;

        location /api {
            set $limit_bucket_capacity 5;
            set $limit_refill_rate 2;

            content_by_lua_block {
                local redis = require "resty.redis"
                local client_ip = ngx.var.remote_addr or "global"
                
                local BUCKET_CAPACITY = tonumber(ngx.var.limit_bucket_capacity) or 5
                local REFILL_RATE = tonumber(ngx.var.limit_refill_rate) or 2
                
                -- 1. Conectar a Redis usando el DNS interno de la red de Docker
                local red = redis:new()
                red:set_timeouts(500, 500, 500)
                
                local ok, err = red:connect("mi-redis-brain", 6379)
                if not ok then
                    ngx.log(ngx.ERR, "[Fail-Safe] Error conectando a Redis: ", err)
                    -- Si falla Redis, dejamos pasar (Fail-safe)
                    ngx.status = ngx.HTTP_OK
                    ngx.say([[{"status": "success", "message": "Peticion aceptada por Fail-Safe (Redis caido)"}]])
                    return ngx.exit(ngx.HTTP_OK)
                end
                
                local tokens_key = client_ip .. ":tokens"
                local time_key = client_ip .. ":last_time"
                local now = ngx.now()
                
                -- 2. Consultar base de datos centralizada
                red:init_pipeline()
                red:get(tokens_key)
                red:get(time_key)
                local results, pipe_err = red:commit_pipeline()
                
                local tokens = results and results[1]
                local last_time = results and results[2]
                
                if tokens == ngx.null or not tokens then
                    tokens = BUCKET_CAPACITY
                    last_time = now
                else
                    tokens = tonumber(tokens)
                    last_time = tonumber(last_time)
                    local time_passed = now - last_time
                    tokens = math.min(BUCKET_CAPACITY, tokens + (time_passed * REFILL_RATE))
                end
                
                -- 3. Reglas del Freno de Mano
                if tokens >= 1 then
                    tokens = tokens - 1
                    
                    red:init_pipeline()
                    red:setex(tokens_key, 3600, tokens)
                    red:setex(time_key, 3600, now)
                    red:commit_pipeline()
                    red:set_keepalive(10000, 100)
                    
                    ngx.status = ngx.HTTP_OK
                    ngx.header.content_type = "application/json; charset=utf-8"
                    ngx.say([[{"status": "success", "message": "Peticion procesada por el cerebro distribuido de Redis!"}]])
                else
                    red:set_keepalive(10000, 100)
                    ngx.status = ngx.HTTP_TOO_MANY_REQUESTS
                    ngx.header.content_type = "application/json; charset=utf-8"
                    ngx.say([[{"error": "Too Many Requests", "message": "Freno de mano: Limite centralizado excedido."}]])
                    return ngx.exit(ngx.HTTP_OK)
                end
            }
        }
    }
}

Overwriting config/nginx.conf


In [22]:
# 1. Destruimos el contenedor problemático anterior
!docker rm -f mi-openresty-gateway 2>/dev/null || true

# 2. Encendemos de nuevo apuntando UNICAMENTE al archivo de configuracion verificado
!docker run -d --name mi-openresty-gateway \
  --network gateway-network \
  -p 8080:8080 \
  -v "$(pwd)/config/nginx.conf:/usr/local/openresty/nginx/conf/nginx.conf:ro" \
  openresty/openresty:alpine

mi-openresty-gateway
6620ddd8119972afc2f61180a1fd29f2b577bcf78becd381a142b8e3afd2baf5


In [24]:
!for i in {1..6}; do curl -s -o /dev/null -w "%{http_code}\n" http://localhost:8080/api; done

200
200
200
200
200
429
